# 06 - Batch Mental Model Comparison (30 Students × 3 Clusters)

Scaled version of experiment 05. Runs controlled A/B comparison 
(Curriculum-Aware WITH vs WITHOUT mental model) across 30 students: 
10 from each cluster (Struggling, Average, High Performer).

In [1]:
import json
import time
import ast
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.cluster.vq import kmeans2
from sklearn.preprocessing import StandardScaler
from google.genai import types

ROOT = Path.cwd()
if not (ROOT / 'lib').exists() and (ROOT.parent / 'lib').exists():
    ROOT = ROOT.parent
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.experiment_utils import create_client, load_best_attempts_df
from lib.llm_batch_analyzer import format_submissions, clean_json_response
from lib.mental_model import (
    load_skill_map, calculate_student_profile, build_prerequisite_graph,
    get_weak_skills, build_mental_model_payload,
)
try:
    from lib.prompt_strategies import build_curriculum_aware_prompt, KC_TAGS
except ModuleNotFoundError:
    from lib.prompts import build_curriculum_aware_prompt, KC_TAGS
from utils.dataset import load_topics_json, load_problem_descriptions

MODEL_ID = 'gemini-2.5-flash'
RANDOM_SEED = 42
PROBLEMS_PER_STUDENT = 12
STUDENTS_PER_CLUSTER = 10
SLEEP_SECONDS = 1.0
WEAK_THRESHOLD = 0.6

client = create_client()
print(f'Ready. Model={MODEL_ID}, {STUDENTS_PER_CLUSTER} students/cluster, {PROBLEMS_PER_STUDENT} problems/student')

Ready. Model=gemini-2.5-flash, 10 students/cluster, 12 problems/student


In [2]:
best_attempts_df = load_best_attempts_df()
skill_map, all_skills = load_skill_map()
all_skills_list = sorted(all_skills)

# Build skill mastery vector for every student
student_ids = best_attempts_df['SubjectID'].unique()
profiles = {}
vectors = {}

for sid in student_ids:
    profile = calculate_student_profile(sid, best_attempts_df, skill_map, all_skills)
    profiles[sid] = profile
    # Build vector (replace None with 0.5 as neutral)
    vec = [profile.get(skill, 0.5) or 0.5 for skill in all_skills_list]
    vectors[sid] = vec

# Create matrix and standardize
sid_list = sorted(vectors.keys())
X = np.array([vectors[sid] for sid in sid_list])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# K-Means clustering (K=3)
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=3, random_state=RANDOM_SEED, n_init=10)
labels = kmeans.fit_predict(X_scaled)

# Assign cluster names by average score
cluster_avg_scores = {}
for i in range(3):
    cluster_sids = [sid_list[j] for j in range(len(sid_list)) if labels[j] == i]
    avg = best_attempts_df[best_attempts_df['SubjectID'].isin(cluster_sids)]['Score'].mean()
    cluster_avg_scores[i] = avg

sorted_clusters = sorted(cluster_avg_scores.items(), key=lambda x: x[1])
cluster_name_map = {
    sorted_clusters[0][0]: "Struggling",
    sorted_clusters[1][0]: "Average",
    sorted_clusters[2][0]: "High Performer",
}

# Assign labels
student_cluster = {}
for i, sid in enumerate(sid_list):
    student_cluster[sid] = cluster_name_map[labels[i]]

# Print cluster sizes
from collections import Counter
cluster_counts = Counter(student_cluster.values())
print("Cluster sizes:")
for name, count in sorted(cluster_counts.items()):
    print(f"  {name}: {count}")

# Known medoids (must be included)
MEDOIDS = {10155: "Struggling", 14359: "Average", 14475: "High Performer"}

Main table: 201,570 rows
CodeState table: 69,627 rows
Subject table: 381 rows
Joined dataset: 191,584 rows
Best attempts: 15,375 rows (372 students, 50 problems)
Cluster sizes:
  Average: 54
  High Performer: 303
  Struggling: 15


In [3]:
np.random.seed(RANDOM_SEED)

selected_students = {}  # {sid: cluster_name}

for cluster_name in ["Struggling", "Average", "High Performer"]:
    # Get all students in this cluster with >= 10 problems
    cluster_sids = [sid for sid, c in student_cluster.items() if c == cluster_name]
    eligible = []
    for sid in cluster_sids:
        n_problems = len(best_attempts_df[best_attempts_df['SubjectID'] == sid])
        if n_problems >= 10:
            eligible.append(sid)

    # Ensure medoid is included
    medoid = [sid for sid, c in MEDOIDS.items() if c == cluster_name][0]
    if medoid not in eligible:
        eligible.append(medoid)

    # Select: medoid + 9 random others
    others = [sid for sid in eligible if sid != medoid]
    np.random.shuffle(others)
    chosen = [medoid] + others[:STUDENTS_PER_CLUSTER - 1]

    for sid in chosen:
        selected_students[sid] = cluster_name

    print(f"{cluster_name}: selected {len(chosen)} students (medoid={medoid})")
    print(f"  IDs: {sorted(chosen)}")

print(f"\nTotal selected: {len(selected_students)}")

Struggling: selected 8 students (medoid=10155)
  IDs: [np.int64(9948), 10155, np.int64(14189), np.int64(14327), np.int64(14374), np.int64(14386), np.int64(14474), np.int64(14499)]
Average: selected 10 students (medoid=14359)
  IDs: [np.int64(106), np.int64(10083), np.int64(10224), np.int64(14186), np.int64(14316), 14359, np.int64(14381), np.int64(14429), np.int64(14459), np.int64(14476)]
High Performer: selected 10 students (medoid=14475)
  IDs: [np.int64(13365), np.int64(14205), np.int64(14289), np.int64(14296), np.int64(14398), np.int64(14407), np.int64(14450), np.int64(14465), np.int64(14471), 14475]

Total selected: 28


In [4]:
topics = load_topics_json() or {}
problem_descriptions = load_problem_descriptions() or {}
G = build_prerequisite_graph()
prompts_df = pd.read_csv(ROOT / 'dataset/CodeWorkout/Problem_Prompts/problem_prompts.csv')

EXACT_KC_TAGS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]
VALID_KC_SET = set(EXACT_KC_TAGS)

# For each student: build mental model, select problems, build prompts
student_data = {}

for sid in selected_students:
    # Mental model
    profile = profiles[sid]
    weak_skills = get_weak_skills(profile, threshold=WEAK_THRESHOLD)
    mental_model = build_mental_model_payload(
        student_id=sid, profile=profile, weak_skill_pairs=weak_skills, graph=G,
    )
    weak_skill_names = [s[0] for s in weak_skills]

    # Select problems (same bucket logic as notebook 05)
    sdf = best_attempts_df[best_attempts_df['SubjectID'] == sid].copy()
    sdf = sdf.drop_duplicates(subset=['ProblemID']).copy()
    score_max = float(sdf['Score'].max())
    if score_max <= 1.0:
        sdf['ScorePct'] = sdf['Score'] * 100.0
    else:
        sdf['ScorePct'] = sdf['Score']

    target_n = min(PROBLEMS_PER_STUDENT, len(sdf))
    full = sdf[sdf['ScorePct'] >= 99.9]
    partial = sdf[(sdf['ScorePct'] > 0) & (sdf['ScorePct'] < 99.9)]
    zero = sdf[sdf['ScorePct'] <= 0]

    base_quota = target_n // 3
    remainder = target_n - (base_quota * 3)
    quotas = {'full': base_quota, 'partial': base_quota, 'zero': base_quota}
    for key in ['partial', 'full', 'zero'][:remainder]:
        quotas[key] += 1

    picked = []
    for name, bucket in [('full', full), ('partial', partial), ('zero', zero)]:
        take_n = min(quotas[name], len(bucket))
        if take_n > 0:
            picked.append(bucket.sample(n=take_n, random_state=RANDOM_SEED))

    if picked:
        selected_df = pd.concat(picked).drop_duplicates(subset=['ProblemID'])
    else:
        selected_df = pd.DataFrame(columns=sdf.columns)

    remaining_needed = target_n - len(selected_df)
    if remaining_needed > 0:
        pool = sdf[~sdf['ProblemID'].isin(selected_df['ProblemID'])]
        if len(pool) > 0:
            topup = pool.sample(n=min(remaining_needed, len(pool)), random_state=RANDOM_SEED)
            selected_df = pd.concat([selected_df, topup])

    selected_df = selected_df.sort_values('ScorePct', ascending=False).head(target_n).reset_index(drop=True)

    # Build prompts
    focus_ids = [int(pid) for pid in selected_df['ProblemID'].tolist()]
    baseline_prompt = build_curriculum_aware_prompt(topics=topics, problems=problem_descriptions, focus_problem_ids=focus_ids)
    enriched_prompt = (
        baseline_prompt
        + "\n\nAdditional Student Mental Model Context:\n"
        + json.dumps(mental_model, indent=2)
        + "\n\nUse this context to better judge likely misconceptions and future risks."
    )

    student_data[sid] = {
        'cluster': selected_students[sid],
        'weak_skills': weak_skill_names,
        'num_weak': len(weak_skill_names),
        'problems_df': selected_df,
        'baseline_prompt': baseline_prompt,
        'enriched_prompt': enriched_prompt,
    }

print(f"Prepared data for {len(student_data)} students")
for cluster_name in ["Struggling", "Average", "High Performer"]:
    cluster_students = [sid for sid, d in student_data.items() if d['cluster'] == cluster_name]
    avg_weak = np.mean([student_data[sid]['num_weak'] for sid in cluster_students])
    avg_problems = np.mean([len(student_data[sid]['problems_df']) for sid in cluster_students])
    print(f"  {cluster_name}: {len(cluster_students)} students, avg {avg_weak:.1f} weak skills, avg {avg_problems:.0f} problems")

Prepared data for 28 students
  Struggling: 8 students, avg 7.0 weak skills, avg 12 problems
  Average: 10 students, avg 0.9 weak skills, avg 12 problems
  High Performer: 10 students, avg 0.0 weak skills, avg 12 problems


In [5]:
def run_one_call(submission_row, system_instruction: str):
    formatted_input = format_submissions([submission_row.to_dict()])
    t0 = time.time()
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=formatted_input,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
                response_mime_type='application/json',
            ),
        )
        raw_text = response.text if response and response.text else '{}'
        parsed = json.loads(clean_json_response(raw_text))
        return parsed, round(time.time() - t0, 3), None
    except Exception as e:
        return None, round(time.time() - t0, 3), str(e)

def extract_expected_kc_tags(problem_id: int) -> list:
    row = prompts_df[prompts_df['ProblemID'] == int(problem_id)]
    if row.empty:
        return []
    row = row.iloc[0]
    return [tag for tag in EXACT_KC_TAGS if pd.notna(row.get(tag, 0)) and float(row.get(tag, 0)) == 1.0]

def extract_kc_tags(output_obj) -> tuple:
    if not isinstance(output_obj, dict):
        return [], []
    valid_tags = set()
    invalid_tags = set()
    analysis_list = output_obj.get("student_analysis", [])
    if not isinstance(analysis_list, list):
        analysis_list = []
    for analysis in analysis_list:
        if not isinstance(analysis, dict): continue
        for gap in analysis.get("knowledge_gaps", []):
            tag = gap.get("missing_concept", "") if isinstance(gap, dict) else str(gap)
            tag = tag.strip()
            if tag:
                (valid_tags if tag in VALID_KC_SET else invalid_tags).add(tag)
        for pred in analysis.get("future_predictions", []):
            tag = pred.get("at_risk_topic", "") if isinstance(pred, dict) else str(pred)
            tag = tag.strip()
            if tag:
                (valid_tags if tag in VALID_KC_SET else invalid_tags).add(tag)
    for key in ["knowledge_gaps", "future_predictions"]:
        val = output_obj.get(key, [])
        if isinstance(val, list):
            for item in val:
                if isinstance(item, str):
                    item = item.strip()
                    if item:
                        (valid_tags if item in VALID_KC_SET else invalid_tags).add(item)
    return sorted(valid_tags), sorted(invalid_tags)

def calculate_overlap(predicted_tags: list, expected_tags: list) -> dict:
    pred_set = set(predicted_tags)
    exp_set = set(expected_tags)
    overlap = pred_set & exp_set
    precision = (len(overlap) / len(pred_set)) if pred_set else 0.0
    recall = (len(overlap) / len(exp_set)) if exp_set else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {'overlap_count': len(overlap), 'precision': round(precision, 4), 'recall': round(recall, 4), 'f1': round(f1, 4)}

def get_gap_count(output_obj):
    try:
        analysis = output_obj.get("student_analysis", [{}]) if isinstance(output_obj, dict) else [{}]
        return len(analysis[0].get("knowledge_gaps", [])) if isinstance(analysis, list) and len(analysis) > 0 else 0
    except:
        return 0

print("Helpers ready.")

Helpers ready.


In [6]:
all_rows = []
student_ids_sorted = sorted(student_data.keys())
total_students = len(student_ids_sorted)
total_calls = total_students * PROBLEMS_PER_STUDENT * 2
call_count = 0

# Create checkpoint directory
checkpoint_dir = ROOT / 'results' / '06_batch_30students'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

start_time = time.time()

for s_idx, sid in enumerate(student_ids_sorted, start=1):
    sd = student_data[sid]
    cluster = sd['cluster']
    weak_names = sd['weak_skills']
    problems_df = sd['problems_df']

    print(f"\n{'='*60}")
    print(f"[{s_idx}/{total_students}] Student {sid} ({cluster}) | {sd['num_weak']} weak skills")
    print(f"{'='*60}")

    for p_idx, (_, sub) in enumerate(problems_df.iterrows(), start=1):
        pid = int(sub['ProblemID'])
        score_pct = float(sub['ScorePct'])
        expected_tags = extract_expected_kc_tags(pid)

        # Condition A: Baseline
        baseline_out, baseline_time, baseline_err = run_one_call(sub, sd['baseline_prompt'])
        call_count += 1

        # Condition B: Enriched
        enriched_out, enriched_time, enriched_err = run_one_call(sub, sd['enriched_prompt'])
        call_count += 1

        # Extract tags
        baseline_tags, baseline_invalid = extract_kc_tags(baseline_out if baseline_err is None else {})
        enriched_tags, enriched_invalid = extract_kc_tags(enriched_out if enriched_err is None else {})

        # Metrics
        baseline_gap_count = get_gap_count(baseline_out)
        enriched_gap_count = get_gap_count(enriched_out)
        baseline_weak = calculate_overlap(baseline_tags, weak_names)
        enriched_weak = calculate_overlap(enriched_tags, weak_names)
        baseline_rel = calculate_overlap(baseline_tags, expected_tags)
        enriched_rel = calculate_overlap(enriched_tags, expected_tags)

        is_perfect = score_pct >= 99.9

        all_rows.append({
            'SubjectID': sid,
            'Cluster': cluster,
            'NumWeakSkills': sd['num_weak'],
            'ProblemID': pid,
            'Score': float(sub['Score']),
            'ScorePct': score_pct,
            'IsPerfect': is_perfect,
            'Baseline_GAP_Count': baseline_gap_count,
            'Enriched_GAP_Count': enriched_gap_count,
            'Baseline_WeakOverlap_F1': baseline_weak['f1'],
            'Enriched_WeakOverlap_F1': enriched_weak['f1'],
            'Baseline_Relevance_F1': baseline_rel['f1'],
            'Enriched_Relevance_F1': enriched_rel['f1'],
            'Baseline_KCTags': baseline_tags,
            'Enriched_KCTags': enriched_tags,
            'Baseline_TimeSec': baseline_time,
            'Enriched_TimeSec': enriched_time,
            'Baseline_PerfectCorrect': (baseline_gap_count == 0) if is_perfect else None,
            'Enriched_PerfectCorrect': (enriched_gap_count == 0) if is_perfect else None,
        })

        tag_status = "SAME" if baseline_tags == enriched_tags else "DIFF"
        elapsed = time.time() - start_time
        eta = (elapsed / call_count) * (total_calls - call_count) / 60
        print(f"  [{p_idx}/12] P{pid} {score_pct:.0f}% | B={baseline_gap_count} E={enriched_gap_count} [{tag_status}] | ETA: {eta:.0f}min")

        if baseline_err:
            print(f"    BASELINE ERROR: {baseline_err}")
        if enriched_err:
            print(f"    ENRICHED ERROR: {enriched_err}")

        time.sleep(SLEEP_SECONDS)

    # Checkpoint every 5 students
    if s_idx % 5 == 0:
        checkpoint_df = pd.DataFrame(all_rows)
        checkpoint_df.to_csv(checkpoint_dir / f'checkpoint_{s_idx}_students.csv', index=False)
        print(f"\n  >>> Checkpoint saved: {s_idx} students, {len(all_rows)} rows <<<")

results_df = pd.DataFrame(all_rows)
elapsed_total = (time.time() - start_time) / 60
print(f"\n{'='*60}")
print(f"COMPLETED: {len(results_df)} rows, {total_students} students, {elapsed_total:.1f} minutes")


[1/28] Student 106 (Average) | 0 weak skills
  [1/12] P1 100% | B=0 E=0 [SAME] | ETA: 37min
  [2/12] P17 100% | B=0 E=0 [SAME] | ETA: 74min
  [3/12] P100 100% | B=0 E=0 [SAME] | ETA: 67min
  [4/12] P3 100% | B=0 E=0 [SAME] | ETA: 63min
  [5/12] P102 100% | B=0 E=0 [SAME] | ETA: 61min
  [6/12] P233 100% | B=0 E=0 [SAME] | ETA: 59min
  [7/12] P5 100% | B=0 E=0 [SAME] | ETA: 57min
  [8/12] P235 100% | B=0 E=0 [SAME] | ETA: 58min
  [9/12] P101 100% | B=0 E=0 [SAME] | ETA: 61min
  [10/12] P22 100% | B=0 E=0 [SAME] | ETA: 61min
  [11/12] P13 100% | B=0 E=0 [SAME] | ETA: 60min
  [12/12] P25 81% | B=2 E=2 [DIFF] | ETA: 80min

[2/28] Student 9948 (Struggling) | 5 weak skills
  [1/12] P41 100% | B=0 E=0 [SAME] | ETA: 77min
  [2/12] P56 100% | B=0 E=0 [SAME] | ETA: 75min
  [3/12] P17 100% | B=0 E=0 [SAME] | ETA: 74min
  [4/12] P67 100% | B=0 E=0 [SAME] | ETA: 72min
  [5/12] P101 96% | B=2 E=1 [SAME] | ETA: 81min
  [6/12] P37 87% | B=1 E=1 [SAME] | ETA: 82min
  [7/12] P22 36% | B=3 E=4 [SAME] | E

In [7]:
results_dir = ROOT / 'results' / '06_batch_30students'
results_dir.mkdir(parents=True, exist_ok=True)

csv_path = results_dir / 'batch_comparison_30students.csv'
results_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
print(f"Columns: {', '.join(results_df.columns)}")

Saved: /mnt/d/Projects/kintsugi/results/06_batch_30students/batch_comparison_30students.csv
Columns: SubjectID, Cluster, NumWeakSkills, ProblemID, Score, ScorePct, IsPerfect, Baseline_GAP_Count, Enriched_GAP_Count, Baseline_WeakOverlap_F1, Enriched_WeakOverlap_F1, Baseline_Relevance_F1, Enriched_Relevance_F1, Baseline_KCTags, Enriched_KCTags, Baseline_TimeSec, Enriched_TimeSec, Baseline_PerfectCorrect, Enriched_PerfectCorrect


In [8]:
print("=" * 70)
print("BATCH RESULTS: 30 Students × 3 Clusters")
print("=" * 70)

for cluster in ["Struggling", "Average", "High Performer"]:
    cdf = results_df[results_df['Cluster'] == cluster]
    fail_df = cdf[cdf['IsPerfect'] == False]
    perf_df = cdf[cdf['IsPerfect'] == True]

    n_students = cdf['SubjectID'].nunique()
    avg_weak = cdf.groupby('SubjectID')['NumWeakSkills'].first().mean()

    print(f"\n{'='*50}")
    print(f"CLUSTER: {cluster} ({n_students} students)")
    print(f"{'='*50}")
    print(f"Avg weak skills: {avg_weak:.1f}")
    print(f"Total problems: {len(cdf)} ({len(perf_df)} perfect, {len(fail_df)} failing)")

    if len(perf_df) > 0:
        b_correct = perf_df['Baseline_PerfectCorrect'].mean() * 100
        e_correct = perf_df['Enriched_PerfectCorrect'].mean() * 100
        print(f"100% problems — Baseline correct: {b_correct:.0f}%, Enriched correct: {e_correct:.0f}%")

    if len(fail_df) > 0:
        b_f1 = fail_df['Baseline_WeakOverlap_F1'].mean()
        e_f1 = fail_df['Enriched_WeakOverlap_F1'].mean()
        delta = e_f1 - b_f1
        print(f"WeakOverlap F1 — Baseline: {b_f1:.3f}, Enriched: {e_f1:.3f}, Delta: {delta:+.3f}")

        b_rel = fail_df['Baseline_Relevance_F1'].mean()
        e_rel = fail_df['Enriched_Relevance_F1'].mean()
        print(f"Relevance F1  — Baseline: {b_rel:.3f}, Enriched: {e_rel:.3f}")

# Overall
print(f"\n{'='*50}")
print(f"OVERALL ({results_df['SubjectID'].nunique()} students)")
print(f"{'='*50}")
fail_all = results_df[results_df['IsPerfect'] == False]
if len(fail_all) > 0:
    b_f1 = fail_all['Baseline_WeakOverlap_F1'].mean()
    e_f1 = fail_all['Enriched_WeakOverlap_F1'].mean()
    print(f"WeakOverlap F1 — Baseline: {b_f1:.3f}, Enriched: {e_f1:.3f}, Delta: {e_f1-b_f1:+.3f}")

    b_rel = fail_all['Baseline_Relevance_F1'].mean()
    e_rel = fail_all['Enriched_Relevance_F1'].mean()
    print(f"Relevance F1  — Baseline: {b_rel:.3f}, Enriched: {e_rel:.3f}")

BATCH RESULTS: 30 Students × 3 Clusters

CLUSTER: Struggling (8 students)
Avg weak skills: 7.0
Total problems: 94 (35 perfect, 59 failing)
100% problems — Baseline correct: 97%, Enriched correct: 89%
WeakOverlap F1 — Baseline: 0.254, Enriched: 0.297, Delta: +0.043
Relevance F1  — Baseline: 0.655, Enriched: 0.631

CLUSTER: Average (10 students)
Avg weak skills: 0.9
Total problems: 116 (71 perfect, 45 failing)
100% problems — Baseline correct: 89%, Enriched correct: 90%
WeakOverlap F1 — Baseline: 0.066, Enriched: 0.128, Delta: +0.061
Relevance F1  — Baseline: 0.571, Enriched: 0.547

CLUSTER: High Performer (10 students)
Avg weak skills: 0.0
Total problems: 120 (107 perfect, 13 failing)
100% problems — Baseline correct: 93%, Enriched correct: 94%
WeakOverlap F1 — Baseline: 0.000, Enriched: 0.000, Delta: +0.000
Relevance F1  — Baseline: 0.551, Enriched: 0.469

OVERALL (28 students)
WeakOverlap F1 — Baseline: 0.153, Enriched: 0.199, Delta: +0.045
Relevance F1  — Baseline: 0.611, Enriched: 0

In [9]:
student_summary = results_df.groupby(['SubjectID', 'Cluster', 'NumWeakSkills']).agg({
    'Baseline_WeakOverlap_F1': 'mean',
    'Enriched_WeakOverlap_F1': 'mean',
    'Baseline_Relevance_F1': 'mean',
    'Enriched_Relevance_F1': 'mean',
    'IsPerfect': 'sum',
}).rename(columns={'IsPerfect': 'PerfectCount'}).reset_index()

student_summary['Delta_F1'] = student_summary['Enriched_WeakOverlap_F1'] - student_summary['Baseline_WeakOverlap_F1']
student_summary = student_summary.sort_values(['Cluster', 'Delta_F1'], ascending=[True, False])

print("Per-Student Summary:")
display(student_summary)

# How many students improved?
improved = (student_summary['Delta_F1'] > 0).sum()
same = (student_summary['Delta_F1'] == 0).sum()
worse = (student_summary['Delta_F1'] < 0).sum()
print(f"\nStudents improved: {improved}, Same: {same}, Worse: {worse}")

# Save
results_dir = ROOT / 'results' / '06_batch_30students'
student_summary.to_csv(results_dir / 'per_student_summary.csv', index=False)

metadata = {
    "experiment": "06_batch_mental_model_comparison",
    "run_timestamp": datetime.now().isoformat(),
    "model_id": MODEL_ID,
    "total_students": int(results_df['SubjectID'].nunique()),
    "total_problems": len(results_df),
    "students_per_cluster": STUDENTS_PER_CLUSTER,
    "overall_baseline_f1": round(float(fail_all['Baseline_WeakOverlap_F1'].mean()), 4) if len(fail_all) > 0 else 0,
    "overall_enriched_f1": round(float(fail_all['Enriched_WeakOverlap_F1'].mean()), 4) if len(fail_all) > 0 else 0,
    "overall_delta_f1": round(float(fail_all['Enriched_WeakOverlap_F1'].mean() - fail_all['Baseline_WeakOverlap_F1'].mean()), 4) if len(fail_all) > 0 else 0,
    "students_improved": int(improved),
    "students_same": int(same),
    "students_worse": int(worse),
}
with open(results_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"\nSaved to: {results_dir}")

Per-Student Summary:


,SubjectID,Cluster,NumWeakSkills,Baseline_WeakOverlap_F1,Enriched_WeakOverlap_F1,Baseline_Relevance_F1,Enriched_Relevance_F1,PerfectCount,Delta_F1
2,10083,Average,1,0.027775,0.126583,0.355058,0.314025,6,0.098808
26,14476,Average,3,0.041667,0.112425,0.460400,0.501367,4,0.070758
13,14359,Average,2,0.051583,0.103167,0.202208,0.273633,7,0.051583
11,14316,Average,1,0.061108,0.094442,0.293083,0.219842,6,0.033333
0,106,Average,0,0.000000,0.000000,0.055558,0.050000,11,0.000000
4,10224,Average,1,0.000000,0.000000,0.226808,0.237850,7,0.000000
15,14381,Average,0,0.000000,0.000000,0.000000,0.000000,10,0.000000
19,14429,Average,0,0.000000,0.000000,0.223340,0.236670,7,0.000000
21,14459,Average,0,0.000000,0.000000,0.405375,0.364108,6,0.000000
6,14186,Average,1,0.108333,0.083992,0.200400,0.231842,7,-0.024342



Students improved: 11, Same: 16, Worse: 1

Saved to: /mnt/d/Projects/kintsugi/results/06_batch_30students


# HUMAN VALIDATION

In [10]:
# The 10 validation cases
cases = [
    (14476, 32),
    (10155, 235),
    (9948, 104),
    (14499, 232),
    (14374, 25),
    (14327, 21),
    (14474, 128),
    (9948, 100),
    (9948, 22),
    (14374, 20),
]

for i, (sid, pid) in enumerate(cases):
    row = best_attempts_df[
        (best_attempts_df['SubjectID'] == sid) & 
        (best_attempts_df['ProblemID'] == pid)
    ]
    if row.empty:
        print(f"Case {i+1}: Student {sid}, Problem {pid} — NOT FOUND")
        continue
    
    code = row.iloc[0]['Code']
    score = row.iloc[0]['Score']
    print(f"{'='*60}")
    print(f"CASE {i+1}: Student {sid}, Problem {pid}, Score {score:.1%}")
    print(f"{'='*60}")
    print(code)
    print()

CASE 1: Student 14476, Problem 32, Score 0.0%
public String plusOut(String str, String word)
{
    if (str.contains(word))
    {
        String newString = "";
        return word;
    }
    else
    {
        return str;
    }
}

CASE 2: Student 10155, Problem 235, Score 85.7%
public int dateFashion(int you, int date)
{
   if (you >= 8 && date >= 8)
   {
       return 2;
   }
    
    if (date >= 8 && you >= 8)
   {
       return 2;
   }
    
    
    else if (you <= 2 || date <= 2)
   {
       return 0;
   }
    
  else if (you >= 2 && date <= 8)
  {
    return 1;
  }
    return 0;
}


CASE 3: Student 9948, Problem 104, Score 33.3%
public int[] zeroMax(int[] nums)
{
    return nums;
}


CASE 4: Student 14499, Problem 232, Score 0.0%
public String alarmClock(int day, boolean vacation)
{
    alarmClock(1, false); return "7:00"
    
}


CASE 5: Student 14374, Problem 25, Score 76.2%
public boolean evenlySpaced(int a, int b, int c)
{   
        if (a - b == b - c)
        {
            r